## Implementing Autoencoder

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam


In [4]:
# !pip install tensorflow

In [5]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier

df= pd.read_csv('./day5_titanic/train.csv')
df.head(2)
y=df['Survived']
X=df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked']]


In [6]:
X_numeric = X[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']]
X_categorical = X[['Sex','Embarked']]
X_neither = X['Cabin']
X_onehot = pd.get_dummies(X_categorical,['Sex','Embarked']).astype(int)
X_cabin_nonna = X['Cabin'].apply(lambda x: 0 if pd.isna(x) else 1)
X_encoded0 = pd.merge(X_numeric, X_cabin_nonna,  left_index=True, right_index=True)
X_encoded1 = pd.merge(X_encoded0, X_onehot,  left_index=True, right_index=True)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded1, y, test_size=0.2, random_state=42, stratify=y
)
from sklearn.impute import SimpleImputer

# Imputer for numeric columns (e.g., Age)
imputer = SimpleImputer(strategy='median')  # or 'mean' if you prefer

# Fit on train, transform train
X_train['Age'] = imputer.fit_transform(X_train[['Age']])

# Transform test using same statistics
X_test['Age'] = imputer.transform(X_test[['Age']])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [7]:
n_features = X_train_scaled.shape[1]

# Define architecture
input_layer = Input(shape=(n_features,))
encoded = Dense(64, activation='relu')(input_layer)
encoded = Dense(32, activation='relu')(encoded)   # latent layer
decoded = Dense(64, activation='relu')(encoded)
output_layer = Dense(n_features, activation='sigmoid')(decoded)

# Autoencoder model
autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

# Train Autoencoder
autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test_scaled, X_test_scaled),
    verbose=0
)


2025-10-26 23:08:33.809030: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
# Encoder model (to get latent features)
encoder = Model(inputs=input_layer, outputs=encoded)

X_train_encoded = encoder.predict(X_train_scaled)
X_test_encoded = encoder.predict(X_test_scaled)


23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


In [9]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_encoded, y_train)

y_pred = clf.predict(X_test_encoded)


In [13]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))
# Confusion matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.7988826815642458

Classification report:
               precision    recall  f1-score   support

           0       0.80      0.90      0.85       110
           1       0.80      0.64      0.71        69

    accuracy                           0.80       179
   macro avg       0.80      0.77      0.78       179
weighted avg       0.80      0.80      0.79       179


Confusion Matrix:
 [[99 11]
 [25 44]]


In [14]:
X_train_encoded

array([[1.3617578 , 0.        , 1.2647432 , ..., 0.54656315, 1.5886617 ,
        1.0521196 ],
       [2.7470903 , 0.        , 1.8558184 , ..., 0.96396905, 2.9893343 ,
        2.0059595 ],
       [3.9521368 , 0.08550488, 3.4625425 , ..., 5.3412127 , 0.97293985,
        5.312054  ],
       ...,
       [0.46047053, 4.2161627 , 1.1536962 , ..., 0.1529828 , 2.3572948 ,
        0.02743835],
       [4.2759457 , 0.        , 2.7195168 , ..., 2.8154726 , 0.84215534,
        2.819821  ],
       [3.458597  , 0.        , 1.9887904 , ..., 1.9063537 , 3.5329435 ,
        2.694267  ]], shape=(712, 32), dtype=float32)

In [15]:
X_test_encoded

array([[1.0205288 , 0.        , 2.3627253 , ..., 2.2711096 , 0.7263509 ,
        1.1211267 ],
       [1.5717885 , 0.        , 0.597025  , ..., 0.15877683, 1.866917  ,
        0.        ],
       [3.7419538 , 1.4383723 , 0.88682145, ..., 1.1891583 , 1.1440163 ,
        2.458703  ],
       ...,
       [2.6611319 , 0.        , 1.8024578 , ..., 1.0221975 , 2.8820667 ,
        1.9759896 ],
       [4.6593256 , 4.827312  , 3.0776396 , ..., 1.9573623 , 0.6903095 ,
        5.7303753 ],
       [1.1586859 , 0.        , 1.4943137 , ..., 1.5893989 , 1.9136708 ,
        0.9864327 ]], shape=(179, 32), dtype=float32)